In [1]:
from openai import OpenAI

In [2]:
openai_client = OpenAI()

In [11]:
def llm(prompt: str) -> str:
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=[{"role": "user", "content": prompt}]
    )
    return response.output_text

In [15]:
question = "I just discovered the course. Can I join?"

In [16]:
data = llm(prompt=question)

In [17]:
print(data)

Absolutely — if the course is still open, you can usually join.

A few quick things to check:
- **Enrollment status:** Is registration still open?
- **Prerequisites:** Do you meet any required background or skills?
- **Format:** Is it self-paced, live, or cohort-based?
- **Access:** Do you need an invite, payment, or application?

If you want, I can help you draft a short message to the instructor or course admin asking whether late enrollment is allowed.


In [9]:
data.output_text

'Yes—very likely.\n\nThe DataTalks.Club **LLM Zoomcamp** is typically open to anyone, and you can usually join even if you’re starting a bit late. You generally just need to:\n\n1. Go to the course page on DataTalks.Club\n2. Register/subscribe if required\n3. Join the community links they provide\n4. Follow along with the materials and homework at your own pace\n\nA couple of notes:\n- It’s often **free**\n- It’s self-paced, so you don’t need to “apply” in the usual sense\n- If the current cohort has already started, you can usually still join and catch up\n\nIf you want, I can help you find the **exact sign-up page** and tell you whether the current cohort is still open.'

In [21]:
context = """
Question: I just discovered the course. Can I join?
Answer: Yes, you can join the course as it starts on 8th June 2026!
"""

In [ ]:
prompt_template = """You are a course FAQ assistant, your role is to answer student queries
You will be asked a question, and you should answer it based on the following context:

{context}

Question: {question}
"""

In [23]:
prompt = prompt_template.format(context=context, question=question)

In [25]:
data = llm(prompt=prompt)
print(data)

Yes, you can join the course as it starts on 8th June 2026!


In [26]:
## Now lets import some docs into memory

In [28]:
import requests
from tqdm.auto import tqdm

In [29]:
courses_url = "https://datatalks.club/faq/json/courses.json"
courses_faq_base_url = "https://datatalks.club/faq"

In [31]:
response = requests.get(courses_url)
courses = response.json()

documents = []
for course in tqdm(courses, desc="Processing courses"):
    course_faq_url = f"{courses_faq_base_url}{course['path']}"
    faq_response = requests.get(course_faq_url)
    faq_response.raise_for_status()
    faq_data = faq_response.json()
    documents.extend(faq_data)

print(len(documents))

Processing courses:   0%|          | 0/6 [00:00<?, ?it/s]

1346


In [32]:
from minsearch import Index

In [33]:
index = Index(
    text_fields = ["question", "answer", "section"],
    keyword_fields = ["course"]
)
index.fit(documents)

In [37]:
index.search(query=question,
             boost_dict={"question": 2.0, "section" : 0.5},
             filter_dict={"course": "data-engineering-zoomcamp"},
             num_results=5)

[{'id': '3f1424af17',
  'course': 'data-engineering-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: Can I still join the course after the start date?',
  'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."},
 {'id': '9e508f2212',
  'course': 'data-engineering-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: When does the course start?',
  'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slac

In [47]:
def build_context(results: list) -> str:
    docs = []
    for result in results:
        docs.append(f"Section: {result['section']}")
        docs.append(f"Question: {result['question']}")
        docs.append(f"Answer: {result['answer']}")
        docs.append(" ")
    return "\n".join(docs).strip()

def search_results(question: str) -> list:
    results = index.search(query=question,
             boost_dict={"question": 2.0, "section" : 0.5},
             filter_dict={"course": "data-engineering-zoomcamp"},
             num_results=5)

    return results

In [48]:
results = search_results(question)
print(build_context(results))

Section: General Course-Related Questions
Question: Course: Can I still join the course after the start date?
Answer: Yes, even if you don't register, you're still eligible to submit the homework.

Be aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.
 
Section: General Course-Related Questions
Question: Course: When does the course start?
Answer: A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).

- Register via the link in the course repo before the cohort starts.
- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.
- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel.
 
Section: General Course-Related Questions
Question: Course - Can I follow the 

In [49]:
prompt_template = """You are a course FAQ assistant, your role is to answer student queries
You will be asked a question, and you should answer it based on the following context, which contains the 
most relevant FAQ entries from the course:

{context}

Question to answer: {question}
"""

In [50]:
def llm_call(prompt: str) -> str:
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=[{"role": "user", "content": prompt}]
    )
    return response.output_text

In [51]:
def llm(question: str) -> str:
    results = search_results(question)
    context = build_context(results)
    prompt = prompt_template.format(context=context, question=question)
    response = llm_call(prompt)
    return response

In [53]:
print(llm(question))

Yes — you can still join the course even after it has started.

If you’re coming in late:
- You’re still eligible to submit the homework, even if you didn’t register on time.
- The course materials stay available, so you can also follow it later at your own pace.

Just keep in mind:
- There are deadlines for homeworks and the final project, so try not to leave everything until the last minute.

If you want, I can also point you to what to set up first before starting.
